<a href="https://colab.research.google.com/github/OuhmadMohamed/DI_Bootcamp/blob/main/Week4/Day3/Exercises_XP_W4_D3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Exercises XP

In [22]:
### download and extract chinook sample DB
import urllib.request
import zipfile
from functools import partial
import os

chinook_url ='/content/chinook.zip'
#chinook_url = 'http://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip'
if not os.path.exists('chinook.zip'):
    print('downloading chinook.zip ', end='')
    with urllib.request.urlopen(chinook_url) as response:
        with open('chinook.zip', 'wb') as f:
            for data in iter(partial(response.read, 4*1024), b''):
                print('.', end='', flush=True);
                f.write(data)

zipfile.ZipFile('chinook.zip').extractall()
assert os.path.exists('chinook/chinook/chinook.db') # Assuming chinook.db is inside a 'chinook' folder

In [23]:
### useful: functions for displaying results from sql queries using pandas
from IPython.display import display
import pandas as pd

def sql(query):
    print()
    print(query)
    print()

def get_results(query):
    global engine
    q = query.statement if isinstance(query, sqlalchemy.orm.query.Query) else query
    return pd.read_sql(q, engine)

def display_results(query):
    df = get_results(query)
    display(df)
    sql(query)



🌟 Exercise 1 : Open the database

In [29]:
import sqlalchemy

# Create engine
engine = sqlalchemy.create_engine("sqlite:///chinook/chinook/chinook.db") # Corrected path to the database

# Create connection
cur = engine.connect()

print("Database connected successfully!")

Database connected successfully!


In [30]:
### useful: extract classes from the chinook database
metadata = sqlalchemy.MetaData()
metadata.reflect(engine)

## we need to do this once
from sqlalchemy.ext.automap import automap_base

# produce a set of mappings from this MetaData.
Base = automap_base(metadata=metadata)

# calling prepare() just sets up mapped classes and relationships.
Base.prepare()

# also prepare an orm session
from sqlalchemy.orm import sessionmaker
Session = sessionmaker(bind=engine)
session = Session()


Exercise 2: Table Names

Print all table names from the database.

In [31]:
print(metadata.tables.keys())

dict_keys(['albums', 'artists', 'customers', 'employees', 'genres', 'invoice_items', 'tracks', 'media_types', 'invoices', 'playlist_track', 'playlists'])


In [32]:
for table in metadata.tables.keys():
    print(table)

albums
artists
customers
employees
genres
invoice_items
tracks
media_types
invoices
playlist_track
playlists


Exercise 3: First Three Tracks

Access the Track class.

In [33]:
Track = Base.classes.tracks

Query first 3 tracks:

In [34]:
tracks = session.query(Track).limit(3)

for track in tracks:
    print(track.TrackId, track.Name)

1 For Those About To Rock (We Salute You)
2 Balls to the Wall
3 Fast As a Shark


Or display with pandas:

In [35]:
display_results(
    session.query(Track).limit(3)
)

,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
1,2,Balls to the Wall,2,2,1,None,342562,5510424,0.99
2,3,Fast As a Shark,3,2,1,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",230619,3990994,0.99



SELECT tracks."TrackId" AS "tracks_TrackId", tracks."Name" AS "tracks_Name", tracks."AlbumId" AS "tracks_AlbumId", tracks."MediaTypeId" AS "tracks_MediaTypeId", tracks."GenreId" AS "tracks_GenreId", tracks."Composer" AS "tracks_Composer", tracks."Milliseconds" AS "tracks_Milliseconds", tracks."Bytes" AS "tracks_Bytes", tracks."UnitPrice" AS "tracks_UnitPrice" 
FROM tracks
 LIMIT ? OFFSET ?



Exercise 4: Track Name and Album Title

Retrieve:

* Track Name
* Album Title

for first 20 tracks.

In [36]:
Track = Base.classes.tracks
Album = Base.classes.albums

query = (
    session.query(
        Track.Name,
        Album.Title
    )
    .join(Album)
    .limit(20)
)

display_results(query)

,Name,Title
0,For Those About To Rock (We Salute You),For Those About To Rock We Salute You
1,Put The Finger On You,For Those About To Rock We Salute You
2,Let's Get It Up,For Those About To Rock We Salute You
3,Inject The Venom,For Those About To Rock We Salute You
4,Snowballed,For Those About To Rock We Salute You
5,Evil Walks,For Those About To Rock We Salute You
6,C.O.D.,For Those About To Rock We Salute You
7,Breaking The Rules,For Those About To Rock We Salute You
8,Night Of The Long Knives,For Those About To Rock We Salute You
9,Spellbound,For Those About To Rock We Salute You



SELECT tracks."Name" AS "tracks_Name", albums."Title" AS "albums_Title" 
FROM tracks JOIN albums ON albums."AlbumId" = tracks."AlbumId"
 LIMIT ? OFFSET ?



Exercise 5: Tracks Sold

Print first 10 sales from invoice_items.

In [37]:
InvoiceItem = Base.classes.invoice_items

query = (
    session.query(InvoiceItem)
    .limit(10)
)

display_results(query)

,InvoiceLineId,InvoiceId,TrackId,UnitPrice,Quantity
0,1,1,2,0.99,1
1,2,1,4,0.99,1
2,3,2,6,0.99,1
3,4,2,8,0.99,1
4,5,2,10,0.99,1
5,6,2,12,0.99,1
6,7,3,16,0.99,1
7,8,3,20,0.99,1
8,9,3,24,0.99,1
9,10,3,28,0.99,1



SELECT invoice_items."InvoiceLineId" AS "invoice_items_InvoiceLineId", invoice_items."InvoiceId" AS "invoice_items_InvoiceId", invoice_items."TrackId" AS "invoice_items_TrackId", invoice_items."UnitPrice" AS "invoice_items_UnitPrice", invoice_items."Quantity" AS "invoice_items_Quantity" 
FROM invoice_items
 LIMIT ? OFFSET ?



Track Names and Quantities Sold

In [38]:
Track = Base.classes.tracks

query = (
    session.query(
        Track.Name,
        InvoiceItem.Quantity
    )
    .join(
        Track,
        InvoiceItem.TrackId == Track.TrackId
    )
    .limit(10)
)

display_results(query)

,Name,Quantity
0,Balls to the Wall,1
1,Restless and Wild,1
2,Put The Finger On You,1
3,Inject The Venom,1
4,Evil Walks,1
5,Breaking The Rules,1
6,Dog Eat Dog,1
7,Overdose,1
8,Love In An Elevator,1
9,Janie's Got A Gun,1



SELECT tracks."Name" AS "tracks_Name", invoice_items."Quantity" AS "invoice_items_Quantity" 
FROM invoice_items JOIN tracks ON invoice_items."TrackId" = tracks."TrackId"
 LIMIT ? OFFSET ?



Exercise 6: Top 10 Tracks Sold

Count how many times each track was sold.

In [39]:
from sqlalchemy import func

query = (
    session.query(
        Track.Name,
        func.sum(InvoiceItem.Quantity).label("TotalSold")
    )
    .join(
        InvoiceItem,
        Track.TrackId == InvoiceItem.TrackId
    )
    .group_by(Track.TrackId)
    .order_by(
        func.sum(InvoiceItem.Quantity).desc()
    )
    .limit(10)
)

display_results(query)

,Name,TotalSold
0,Balls to the Wall,2
1,Inject The Venom,2
2,Snowballed,2
3,Overdose,2
4,Deuces Are Wild,2
5,Not The Doctor,2
6,Por Causa De Você,2
7,Welcome Home (Sanitarium),2
8,Snowblind,2
9,Cornucopia,2



SELECT tracks."Name" AS "tracks_Name", sum(invoice_items."Quantity") AS "TotalSold" 
FROM tracks JOIN invoice_items ON tracks."TrackId" = invoice_items."TrackId" GROUP BY tracks."TrackId" ORDER BY sum(invoice_items."Quantity") DESC
 LIMIT ? OFFSET ?



Exercise 7: Top 10 Highest Selling Artists

Get ORM Classes

In [40]:
Artist = Base.classes.artists
Album = Base.classes.albums
Track = Base.classes.tracks
InvoiceItem = Base.classes.invoice_items

Query Top Artists

In [41]:
query = (
    session.query(
        Artist.Name,
        func.sum(InvoiceItem.Quantity).label("TotalSales")
    )
    .join(
        Album,
        Artist.ArtistId == Album.ArtistId
    )
    .join(
        Track,
        Album.AlbumId == Track.AlbumId
    )
    .join(
        InvoiceItem,
        Track.TrackId == InvoiceItem.TrackId
    )
    .group_by(Artist.ArtistId)
    .order_by(
        func.sum(InvoiceItem.Quantity).desc()
    )
    .limit(10)
)

display_results(query)

,Name,TotalSales
0,Iron Maiden,140
1,U2,107
2,Metallica,91
3,Led Zeppelin,87
4,Os Paralamas Do Sucesso,45
5,Deep Purple,44
6,Faith No More,42
7,Lost,41
8,Eric Clapton,40
9,R.E.M.,39



SELECT artists."Name" AS "artists_Name", sum(invoice_items."Quantity") AS "TotalSales" 
FROM artists JOIN albums ON artists."ArtistId" = albums."ArtistId" JOIN tracks ON albums."AlbumId" = tracks."AlbumId" JOIN invoice_items ON tracks."TrackId" = invoice_items."TrackId" GROUP BY artists."ArtistId" ORDER BY sum(invoice_items."Quantity") DESC
 LIMIT ? OFFSET ?



Bonus: Top Artists by Revenue

Instead of quantity sold, calculate revenue.

In [42]:
query = (
    session.query(
        Artist.Name,
        func.sum(
            InvoiceItem.Quantity *
            InvoiceItem.UnitPrice
        ).label("Revenue")
    )
    .join(
        Album,
        Artist.ArtistId == Album.ArtistId
    )
    .join(
        Track,
        Album.AlbumId == Track.AlbumId
    )
    .join(
        InvoiceItem,
        Track.TrackId == InvoiceItem.TrackId
    )
    .group_by(Artist.ArtistId)
    .order_by(
        func.sum(
            InvoiceItem.Quantity *
            InvoiceItem.UnitPrice
        ).desc()
    )
    .limit(10)
)

display_results(query)

,Name,Revenue
0,Iron Maiden,138.60
1,U2,105.93
2,Metallica,90.09
3,Led Zeppelin,86.13
4,Lost,81.59
5,The Office,49.75
6,Os Paralamas Do Sucesso,44.55
7,Deep Purple,43.56
8,Faith No More,41.58
9,Eric Clapton,39.60



SELECT artists."Name" AS "artists_Name", sum(invoice_items."Quantity" * invoice_items."UnitPrice") AS "Revenue" 
FROM artists JOIN albums ON artists."ArtistId" = albums."ArtistId" JOIN tracks ON albums."AlbumId" = tracks."AlbumId" JOIN invoice_items ON tracks."TrackId" = invoice_items."TrackId" GROUP BY artists."ArtistId" ORDER BY sum(invoice_items."Quantity" * invoice_items."UnitPrice") DESC
 LIMIT ? OFFSET ?

